# Comparação entre experimentos: BFI-10 (piloto) vs. BFI-44 (multi-traço e traço dominante)

Este notebook compara três datasets, todos gerados com `llama3.1:8b`:

1. **Piloto** (`dataset_llama31_8b.csv`) — BFI-10, 5 traços simultâneos (desenho original, é a análise principal atual do TCC até este ponto).
2. **Experimento A** (`exp_a_bfi44_multitraco_8b.csv`) — BFI-44, 5 traços simultâneos. Mesmo desenho do piloto, só muda o instrumento de checagem (10 → 44 itens).
3. **Experimento B** (`exp_b_bfi44_dominante_8b.csv`) — BFI-44, 1 traço dominante por agente (os outros 4 fixos em Médio).

**Duas comparações, dois propósitos diferentes:**
- **Piloto vs. Experimento A** → efeito de trocar o instrumento de checagem de manipulação (BFI-10 → BFI-44), mantendo o modelo e o desenho de indução constantes.
- **Experimento A vs. Experimento B** → efeito de induzir os 5 traços simultaneamente vs. isolar 1 traço por vez (preocupação metodológica levantada pelo orientador: será que manipular vários traços ao mesmo tempo dilui/confunde o sinal de cada um?).

Rode este notebook só depois que os dois experimentos (A e B) já tiverem os 300 agentes completos e salvos em `resultados/`.

In [ ]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm

DIMS = ['Extroversão', 'Afabilidade', 'Consciência', 'Neuroticismo', 'Abertura à Experiência']
PERFIS_VALIDOS = ['Conservador', 'Moderado', 'Arrojado']


def carregar(caminho):
    df = pd.read_csv(caminho)
    df = df[df['ipa_perfil'].isin(PERFIS_VALIDOS)].copy()
    q1_pts = df['ipa_q1_atribuida'].map({'a': 4, 'b': 3, 'c': 2, 'd': 1})
    df['ipa_score_sem_q1'] = df['ipa_score'] - q1_pts
    return df


df_piloto = carregar('resultados/piloto/dataset_llama31_8b.csv')
df_a = carregar('resultados/principal/exp_a_bfi44_multitraco_8b.csv')
df_b = carregar('resultados/principal/exp_b_bfi44_dominante_8b.csv')

print(f'Piloto (BFI-10):        N={len(df_piloto)}')
print(f'Experimento A (BFI-44): N={len(df_a)}')
print(f'Experimento B (BFI-44): N={len(df_b)}')

## 1. Piloto (BFI-10) vs. Experimento A (BFI-44) — efeito do instrumento

Mesmo modelo (`llama3.1:8b`), mesmo desenho de indução (5 traços simultâneos), só muda o instrumento de checagem de manipulação. Se os números baterem de forma parecida, o instrumento (10 vs. 44 itens) não parece estar mudando a conclusão substantiva do trabalho.

In [ ]:
def tabela_comparativa(dfs_nomeados, coluna_valor, titulo):
    print(titulo)
    print(f'{"Traço":<26}' + ''.join(f'{nome:>18}' for nome, _ in dfs_nomeados))
    for d in DIMS:
        linha = f'{d:<26}'
        for _, df in dfs_nomeados:
            if coluna_valor == 'manipulacao':
                col_nivel, col_alvo = f'nivel_{d}_num', f'bfi_{d}'
            else:
                col_nivel, col_alvo = f'nivel_{d}_num', 'ipa_score_sem_q1'
            r, p = stats.spearmanr(df[col_nivel], df[col_alvo])
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
            linha += f'{r:>+11.3f} {sig:>5}'
        print(linha)
    print()


tabela_comparativa(
    [('Piloto (BFI-10)', df_piloto), ('Exp. A (BFI-44)', df_a)],
    'manipulacao',
    'Checagem de manipulação (nível induzido × autorrelato):',
)
tabela_comparativa(
    [('Piloto (BFI-10)', df_piloto), ('Exp. A (BFI-44)', df_a)],
    'ipa',
    'Correlação de Spearman (nível induzido × escore IPA):',
)

In [ ]:
def rodar_ols(df, rotulo):
    X = sm.add_constant(df[[f'nivel_{d}_num' for d in DIMS]])
    modelo = sm.OLS(df['ipa_score_sem_q1'], X).fit()
    print(f'{rotulo}: R²={modelo.rsquared:.4f}, F={modelo.fvalue:.3f}, p={modelo.f_pvalue:.4f}')
    for d in DIMS:
        coef = modelo.params[f'nivel_{d}_num']
        p = modelo.pvalues[f'nivel_{d}_num']
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
        print(f'  {d:<26}: coef={coef:+.4f} p={p:.4f} {sig}')
    print()
    return modelo


print('=== OLS -- Piloto (BFI-10) ===')
modelo_piloto = rodar_ols(df_piloto, 'Piloto')
print('=== OLS -- Experimento A (BFI-44) ===')
modelo_a = rodar_ols(df_a, 'Experimento A')

**Leitura desta seção**: compare, traço a traço, se a direção (sinal), a significância e a magnitude aproximada dos coeficientes se mantêm entre o piloto e o Experimento A. Preste atenção especial em Abertura à Experiência (achado principal) e Extroversão (achado exploratório) -- se algum desaparecer ou mudar de direção no BFI-44, isso é algo importante para discutir no TCC, não para ignorar.

## 2. Experimento A vs. Experimento B — traços simultâneos diluem o sinal?

Mesmo modelo e instrumento (`llama3.1:8b`, BFI-44); muda só o desenho de indução. No Experimento B, cada agente só tem 1 traço variando ("dominante"), os outros 4 ficam fixos em Médio. A comparação relevante aqui **não** é a OLS com os 5 traços simultâneos (não se aplica da mesma forma no Experimento B — ver aviso no notebook do Colab), e sim: para cada traço, a correlação nível×autorrelato (e nível×IPA) **no Experimento A** (com os outros 4 traços variando) **vs. no Experimento B, só entre os agentes onde aquele traço era o dominante** (com os outros 4 travados em Médio).

In [ ]:
print('Checagem de manipulação -- Exp. A (5 traços variando) vs. Exp. B (só o dominante variando):')
print(f'{"Traço":<26}{"Exp. A":>18}{"Exp. B (dominante)":>22}{"N (B)":>8}')
for d in DIMS:
    r_a, p_a = stats.spearmanr(df_a[f'nivel_{d}_num'], df_a[f'bfi_{d}'])
    sig_a = '***' if p_a < 0.001 else ('**' if p_a < 0.01 else ('*' if p_a < 0.05 else 'n.s.'))

    sub_b = df_b[df_b['traco_dominante'] == d]
    if len(sub_b) >= 5:
        r_b, p_b = stats.spearmanr(sub_b[f'nivel_{d}_num'], sub_b[f'bfi_{d}'])
        sig_b = '***' if p_b < 0.001 else ('**' if p_b < 0.01 else ('*' if p_b < 0.05 else 'n.s.'))
        b_str = f'{r_b:+.3f} {sig_b}'
    else:
        b_str = '(N insuficiente)'
    print(f'{d:<26}{r_a:>+14.3f} {sig_a:>3}{b_str:>22}{len(sub_b):>8}')
print()

print('Correlação com escore IPA -- Exp. A vs. Exp. B (dominante):')
print(f'{"Traço":<26}{"Exp. A":>18}{"Exp. B (dominante)":>22}{"N (B)":>8}')
for d in DIMS:
    r_a, p_a = stats.spearmanr(df_a[f'nivel_{d}_num'], df_a['ipa_score_sem_q1'])
    sig_a = '***' if p_a < 0.001 else ('**' if p_a < 0.01 else ('*' if p_a < 0.05 else 'n.s.'))

    sub_b = df_b[df_b['traco_dominante'] == d]
    if len(sub_b) >= 5:
        r_b, p_b = stats.spearmanr(sub_b[f'nivel_{d}_num'], sub_b['ipa_score_sem_q1'])
        sig_b = '***' if p_b < 0.001 else ('**' if p_b < 0.01 else ('*' if p_b < 0.05 else 'n.s.'))
        b_str = f'{r_b:+.3f} {sig_b}'
    else:
        b_str = '(N insuficiente)'
    print(f'{d:<26}{r_a:>+14.3f} {sig_a:>3}{b_str:>22}{len(sub_b):>8}')

**Como interpretar**:
- Se a correlação de cada traço for **parecida** entre o Experimento A e o Experimento B (dominante), isso é evidência de que induzir os 5 traços ao mesmo tempo **não** está diluindo/confundindo o sinal -- reforça manter o desenho atual (igual à lógica dos artigos-base, onde cada sujeito tem os 5 traços presentes).
- Se a correlação for **sistematicamente mais forte no Experimento B** (traço isolado) do que no A (5 traços juntos), isso é evidência a favor da preocupação do orientador -- vale discutir como limitação do desenho principal e considerar reportar o Experimento B como validação adicional no TCC.
- Atenção ao **N por traço no Experimento B** (coluna da direita): com 300 agentes e 5 traços possíveis, cada traço tem só ~60 agentes onde foi o dominante -- números com N pequeno têm intervalo de confiança largo, não interprete diferenças pequenas como conclusivas.

## 3. Resumo para o texto do TCC

Depois de rodar as células acima, preencha um resumo aqui (ex.: "Abertura à Experiência permaneceu significativa e na mesma direção nos três datasets [piloto/A/B]; o efeito de Extroversão desapareceu no Experimento B, sugerindo que...") -- isso vira a base do texto que substitui os `\FIXME` correspondentes no Capítulo 3 (nota de que BFI-44 virou a análise principal) e no Capítulo 5 (discussão do Experimento B).